<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/03_rev_and_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 3: The Crisis of Confidence — Representative Elementary Volume (REV)

**The Pain Point:** You spend hours simulating the transport properties of a $200^3$ voxel crop from a micro-CT scan and publish the result. Reviewer 2 asks: *"How do you know that specific crop is representative of the whole battery?"* If you had cropped a different section, the results would change. Slower tools cannot run enough iterations to prove statistical confidence, leaving researchers guessing.

**The Solution:** We use OpenImpala’s massive parallel speed to compute the tortuosity of **hundreds** of random sub-volumes in seconds, plotting the statistical variance to find the true Representative Elementary Volume (REV) plateau.

**In this tutorial we will:**
1. Download a real micro-CT scan of a battery electrode.
2. Extract over 100 random sub-volumes of varying sizes (from $16^3$ up to $80^3$).
3. Solve the 3D physics for every single crop.
4. Plot an advanced "Violin REV Trumpet" to visualize the probability density of our results.
5. Calculate the Coefficient of Variation (CV) to mathematically define our REV.
6. Reveal how this physical "uncertainty" is actually the secret weapon for training AI surrogate models.

In [ ]:
# Install OpenImpala and utilities (takes < 5 seconds)
!pip install -q openimpala tifffile matplotlib

In [ ]:
import os
import time
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import openimpala as oi

print(f"🚀 OpenImpala version {oi.__version__} loaded.")

# --- THE JUPYTER HPC FIX ---
# Keep AMReX and MPI alive for the entire notebook to prevent kernel crashes
global_session = oi.Session()
global_session.__enter__()

# --- THE COLAB DATA FIX ---
# Download the sample TIFF directly from the OpenImpala GitHub repo
data_url = "https://raw.githubusercontent.com/BASE-Laboratory/OpenImpala/master/data/SampleData_2Phase_stack_3d_1bit.tif"
data_path = "SampleData_2Phase_stack_3d_1bit.tif"

if not os.path.exists(data_path):
    print("Downloading sample XRT CT scan...")
    urllib.request.urlretrieve(data_url, data_path)
    print("Download complete!")

# Load the scan into memory
microstructure = tifffile.imread(data_path).astype(np.int32)
print(f"Base microstructure loaded: {microstructure.shape}")

## 1. Setting Up the High-Throughput Statistics Loop

If you pick a tiny $16^3$ box, the tortuosity will vary wildly depending on whether you happen to land inside a large pore or a dense solid cluster. As the box size increases, the physical properties should average out and stabilize.

We are going to define a list of window sizes. For each size, we will randomly crop the microstructure 20 times. **Crucially, we will calculate BOTH the local porosity and the local tortuosity for every crop.**

*Note: Because OpenImpala is so fast, we can run this entire statistical mechanics study interactively.*

In [ ]:
window_sizes =[16, 24, 32, 40, 48, 64, 80]
n_samples_per_size = 20  # Total of 140 physics simulations!

# We will store the results in a dictionary containing lists of both Tau and Porosity
rev_results = {w: {'tau': [], 'vf':[]} for w in window_sizes}

print(f"Running {len(window_sizes) * n_samples_per_size} 3D simulations...")
t_start = time.time()
total_solves = 0

with oi.Session():
    for w in window_sizes:
        max_start_idx = microstructure.shape[0] - w
        
        for _ in range(n_samples_per_size):
            # 1. Pick a random starting corner for our crop
            z = np.random.randint(0, max_start_idx + 1)
            y = np.random.randint(0, max_start_idx + 1)
            x = np.random.randint(0, max_start_idx + 1)
            
            # 2. Extract the sub-volume
            crop = microstructure[z:z+w, y:y+w, x:x+w]
            
            # 3. Calculate local Porosity (Volume Fraction of Phase 1)
            vf = oi.volume_fraction(crop, phase=1).fraction
            
            # 4. Fast Percolation Check
            perc = oi.percolation_check(crop, phase=1, direction="z")
            
            # 5. If it percolates, solve the Laplacian!
            if perc.percolates:
                res = oi.tortuosity(crop, phase=1, direction="z", solver="flexgmres")
                rev_results[w]['tau'].append(res.tortuosity)
                rev_results[w]['vf'].append(vf)
                total_solves += 1

t_end = time.time()

# --- THE SPEED FLEX ---
print(f"\n✅ Solved {total_solves} physical 3D microstructures in {t_end - t_start:.2f} seconds.")

## 2. The Probability Density "Trumpet"

A mean and a standard deviation are nice, but they assume your data is normally distributed. By rendering a **Violin Plot**, we can see the actual *probability density* of the physics at each scale. 

Notice how at small sizes, the distribution is extremely skewed (a long tail of high-tortuosity bottlenecks). As the window grows, it collapses into a tight Gaussian distribution.

In [ ]:
means = []
stds =[]
valid_sizes = []
plot_data =[]

# Extract data, ignoring sizes where nothing percolated
for w in window_sizes:
    taus = rev_results[w]['tau']
    if len(taus) > 0:
        means.append(np.mean(taus))
        stds.append(np.std(taus))
        valid_sizes.append(w)
        plot_data.append(taus)

means = np.array(means)
stds = np.array(stds)
valid_sizes = np.array(valid_sizes)

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

# Render the Violin Plot
parts = ax.violinplot(plot_data, positions=valid_sizes, widths=5, 
                      showmeans=False, showextrema=False)

# Style the violins
for pc in parts['bodies']:
    pc.set_facecolor('#1f77b4')
    pc.set_edgecolor('black')
    pc.set_alpha(0.5)

# Plot the Mean trend line
ax.plot(valid_sizes, means, color='#d62728', linewidth=2.5, marker='o', markersize=6, label='Mean Tortuosity')

ax.set_xlabel("Window Size (Voxels)", fontsize=12, fontweight='bold')
ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontsize=12, fontweight='bold')
ax.set_title("Probability Density of Microstructural Transport (REV)", fontsize=14, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend()

plt.tight_layout()
plt.show()

## 3. Rigorous Uncertainty Quantification

To actually prove to Reviewer 2 that we have reached a representative volume, we need to show that the relative error drops below an acceptable threshold (usually 5%). We measure this using the **Coefficient of Variation (CV)**: $CV = \frac{\sigma}{\mu} \times 100\%$.

Additionally, let's look at *why* that variance exists at small scales. By plotting our local Porosity vs. our local Tortuosity for a small sub-volume, we can see the direct physical correlation.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

# --- SUBPLOT 1: Coefficient of Variation ---
cvs = (stds / means) * 100
ax1.plot(valid_sizes, cvs, color='#2ca02c', linewidth=2.5, marker='s', markersize=7)
ax1.axhline(y=5.0, color='r', linestyle='--', linewidth=2, label='5% Threshold (REV Limit)')

ax1.set_xlabel("Window Size (Voxels)", fontweight='bold')
ax1.set_ylabel("Coefficient of Variation (%)", fontweight='bold')
ax1.set_title("Defining the REV Threshold", fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- SUBPLOT 2: The Physical Origin of Variance ---
# Let's look at the sub-volumes from the W=32 window size
w_small = 32
local_vfs = np.array(rev_results[w_small]['vf']) * 100 # Convert to percentage
local_taus = np.array(rev_results[w_small]['tau'])

ax2.scatter(local_vfs, local_taus, color='#ff7f0e', edgecolor='black', s=60, alpha=0.8)

# Add a linear trendline
if len(local_vfs) > 1:
    z = np.polyfit(local_vfs, local_taus, 1)
    p = np.poly1d(z)
    ax2.plot(local_vfs, p(local_vfs), "k-", alpha=0.6, linewidth=2)

ax2.set_xlabel("Local Porosity (%)", fontweight='bold')
ax2.set_ylabel(r"Local Tortuosity ($\tau$)", fontweight='bold')
ax2.set_title(f"Physical Correlation (Scale: {w_small}^3)", fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 4. The Pivot: From Uncertainty to Artificial Intelligence

Look at the right-hand plot above. Because we sampled tiny $32^3$ regions, the porosity varied wildly from crop to crop. Traditional materials science views this variance as a problem: *"This box is too small, throw the data away, it's too noisy."*

But look closely at the trend. That "noise" represents the actual, highly-correlated physical reality of how local geometric features (tight bottlenecks vs. wide open pores) affect transport resistance.

**In the world of Machine Learning, this variance isn't a bug—it's a dataset.**

In the next tutorial (**The AI Data Factory**), we will use OpenImpala to intentionally generate thousands of these small, highly-variable crops. We will use them to train a Random Forest Surrogate Model that learns the exact mapping between local morphology and tortuosity—allowing us to predict physics 1,000x faster.